# MVC Project - MLP Implementation
**Roll No:** 24I-0680  
**Course:** Artificial Neural Networks  
**Dataset:** MNIST Handwritten Digits  
**Architecture:** 784 → 128 → 64 → 10 (Sigmoid activations, NumPy only)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
print('Libraries loaded.')

In [ ]:
# ── Load MNIST (downloads automatically if not found) ──────────────────────
import os, urllib.request

DATA_PATH = 'data/mnist.npz'
URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'

if not os.path.exists(DATA_PATH):
    print('Downloading MNIST ...')
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve(URL, DATA_PATH)
    print('Done.')
else:
    print('MNIST found at', DATA_PATH)

with np.load(DATA_PATH) as d:
    X_train_raw = d['x_train']   # (60000, 28, 28)
    y_train_raw = d['y_train']   # (60000,)
    X_test_raw  = d['x_test']    # (10000, 28, 28)
    y_test_raw  = d['y_test']    # (10000,)

print(f'Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}')

In [ ]:
# ── Pre-processing ──────────────────────────────────────────────────────────
# Flatten images and normalise pixel values to [0, 1]
X_train = X_train_raw.reshape(60000, 784).astype('float64') / 255.0
X_test  = X_test_raw.reshape(10000, 784).astype('float64')  / 255.0

# One-hot encode labels
def one_hot(labels, num_classes=10):
    """Convert integer labels to one-hot matrix."""
    ohe = np.zeros((labels.shape[0], num_classes), dtype='float64')
    ohe[np.arange(labels.shape[0]), labels] = 1.0
    return ohe

Y_train = one_hot(y_train_raw)  # (60000, 10)
Y_test  = one_hot(y_test_raw)   # (10000, 10)

print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'X_test : {X_test.shape}   Y_test : {Y_test.shape}')

In [ ]:
# ── Weight Initialisation ───────────────────────────────────────────────────
# Weights ~ Uniform(-0.5, 0.5),  Biases = 0
np.random.seed(42)

W1 = np.random.uniform(-0.5, 0.5, (784, 128))
b1 = np.zeros((1, 128))

W2 = np.random.uniform(-0.5, 0.5, (128, 64))
b2 = np.zeros((1, 64))

W3 = np.random.uniform(-0.5, 0.5, (64, 10))
b3 = np.zeros((1, 10))

print('Weights initialised:')
print(f'  W1: {W1.shape}  b1: {b1.shape}')
print(f'  W2: {W2.shape}  b2: {b2.shape}')
print(f'  W3: {W3.shape}  b3: {b3.shape}')

In [ ]:
# ── Step 1 : Sigmoid Activation (from Task 1) ───────────────────────────────
def sigmoid(z):
    """Element-wise sigmoid: σ(z) = 1 / (1 + e^{-z}).
    Clipping prevents overflow on very large/small z.
    """
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(a):
    """Derivative of sigmoid given its output a = σ(z).
    σ'(z) = σ(z)(1 - σ(z)) = a(1 - a)
    """
    return a * (1.0 - a)

# Quick sanity check
print('σ(0)   =', sigmoid(0))          # should be 0.5
print('σ\'(0.5) =', sigmoid_derivative(0.5))  # should be 0.25

In [ ]:
# ── Step 2 : Forward Pass (from Task 1) ────────────────────────────────────
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    """Feed X through the 3-layer MLP.

    Returns all intermediate activations needed for backprop.
    """
    # Layer 1  (784 → 128, Sigmoid)
    Z1 = X @ W1 + b1          # weighted sum
    A1 = sigmoid(Z1)           # activation

    # Layer 2  (128 → 64, Sigmoid)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # Output layer  (64 → 10, Sigmoid)
    Z3 = A2 @ W3 + b3
    A3 = sigmoid(Z3)           # predictions (batch, 10)

    return Z1, A1, Z2, A2, Z3, A3

In [ ]:
# ── Step 3 : MSE Loss (from Task 2) ────────────────────────────────────────
def mse_loss(Y_true, Y_pred):
    """Mean Squared Error over all samples and output neurons.

    L = (1/m) * sum_i sum_k (y_ik - ŷ_ik)^2  / K
    (np.mean handles both dimensions automatically)
    """
    return np.mean((Y_true - Y_pred) ** 2)

In [ ]:
# ── Step 4 : Backpropagation (from Task 3) ─────────────────────────────────
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3):
    """Compute gradients of MSE loss w.r.t. all weights and biases.

    Follows the same delta notation derived by hand in Task 3B.
    """
    m = X.shape[0]  # batch size

    # ── Output layer delta ────────────────────────────────────────────────
    # ∂L/∂ŷ = -2(y - ŷ)/m  (per element, MSE derivative)
    # δ3 = ∂L/∂ŷ · σ'(Z3) = -2(Y-A3) · A3(1-A3)
    delta3 = -2 * (Y_true - A3) * sigmoid_derivative(A3) / m

    # Gradients for W3, b3
    dW3 = A2.T @ delta3            # (64, 10)
    db3 = np.sum(delta3, axis=0, keepdims=True)  # (1, 10)

    # ── Hidden Layer 2 delta ──────────────────────────────────────────────
    # δ2 = (δ3 @ W3.T) · σ'(Z2)
    delta2 = (delta3 @ W3.T) * sigmoid_derivative(A2)

    dW2 = A1.T @ delta2            # (128, 64)
    db2 = np.sum(delta2, axis=0, keepdims=True)  # (1, 64)

    # ── Hidden Layer 1 delta ──────────────────────────────────────────────
    # δ1 = (δ2 @ W2.T) · σ'(Z1)
    delta1 = (delta2 @ W2.T) * sigmoid_derivative(A1)

    dW1 = X.T @ delta1             # (784, 128)
    db1 = np.sum(delta1, axis=0, keepdims=True)  # (1, 128)

    return dW1, db1, dW2, db2, dW3, db3

In [ ]:
# ── Step 5 : Weight Update (from Task 4) ───────────────────────────────────
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    """Gradient descent update: w ← w - η·∂L/∂w"""
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3
    return W1, b1, W2, b2, W3, b3

In [ ]:
# ── Training Loop (Mini-Batch GD, Task 5 style) ────────────────────────────
learning_rate = 0.1
epochs        = 20
batch_size    = 32

loss_history  = []
n_train       = X_train.shape[0]

for epoch in range(1, epochs + 1):
    # Shuffle training data each epoch
    idx      = np.random.permutation(n_train)
    X_shuf   = X_train[idx]
    Y_shuf   = Y_train[idx]

    # Mini-batch updates
    for start in range(0, n_train, batch_size):
        X_batch = X_shuf[start : start + batch_size]
        Y_batch = Y_shuf[start : start + batch_size]

        # Forward pass
        Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3)

        # Backpropagation
        dW1, db1, dW2, db2, dW3, db3 = backpropagation(
            X_batch, Y_batch, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3)

        # Update weights
        W1, b1, W2, b2, W3, b3 = update_weights(
            W1, b1, W2, b2, W3, b3,
            dW1, db1, dW2, db2, dW3, db3,
            learning_rate)

    # Compute full-epoch loss for logging
    _, _, _, _, _, A3_full = forward_pass(X_train, W1, b1, W2, b2, W3, b3)
    epoch_loss = mse_loss(Y_train, A3_full)
    loss_history.append(epoch_loss)
    print(f'Epoch {epoch:2d}/{epochs}  |  MSE Loss: {epoch_loss:.6f}')

print('\nTraining complete.')

In [ ]:
# ── Output 1 : Loss Curve ──────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(range(1, epochs + 1), loss_history,
         marker='o', linewidth=2, color='steelblue', markersize=5)
plt.title('Training Loss (MSE) vs Epoch — 24I-0680', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.xticks(range(1, epochs + 1))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print(f'Final loss: {loss_history[-1]:.6f}')

In [ ]:
# ── Output 2 : Test Accuracy ───────────────────────────────────────────────
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)

predicted = np.argmax(A3_test,  axis=1)   # predicted class
true_lbls = np.argmax(Y_test,   axis=1)   # true class

accuracy = np.mean(predicted == true_lbls) * 100.0
print(f'Test Accuracy: {accuracy:.2f}%  ({int(accuracy*100)} / 10000 correct)')

In [ ]:
# ── Output 3 : Sample Predictions (one per digit class 0-9) ───────────────
sample_idx = [np.where(true_lbls == d)[0][0] for d in range(10)]

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
fig.suptitle('Sample Predictions — one per digit class (green=correct, red=wrong)',
             fontsize=12)

for ax, idx in zip(axes.flatten(), sample_idx):
    img   = X_test[idx].reshape(28, 28)
    true  = true_lbls[idx]
    pred  = predicted[idx]
    color = 'green' if true == pred else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'True: {true}  Pred: {pred}', fontsize=10, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150)
plt.show()